In [ ]:
# retried
import openreview
import pandas as pd

def discover_iclr_invitations():
    years = range(2026, 2027)
    
    # The suffixes to test, as requested
    suffixes = [
        'submission', 
        'Submission', 
        'Blind_Submission', 
        'Withdrawn_Submission', 
        'Rejected_Submission', 
        'Desk_Rejected_Submission',
        '' # Test for exact ID match without suffix if needed
    ]

    # Initialize clients
    client_v1 = openreview.Client(baseurl='https://api.openreview.net')
    client_v2 = openreview.api.OpenReviewClient(baseurl='https://api2.openreview.net')

    valid_invitations = []

    print(f"{'Year':<6} | {'API':<4} | {'Status':<10} | {'Invitation ID'}")
    print("-" * 80)

    for year in years:
        # Determine API version
        api_version = 1 if year < 2024 else 2
        client = client_v1 if api_version == 1 else client_v2
        
        # Determine base prefix
        # 2017 is unique with lowercase 'conference'
        if year == 2017:
            prefix = f"ICLR.cc/{year}/conference/-/"
        else:
            prefix = f"ICLR.cc/{year}/Conference/-/"

        for suffix in suffixes:
            # Construct full invitation ID
            # Handle empty suffix case to avoid trailing /
            if suffix == '':
                full_id = prefix.rstrip('/-') 
            else:
                full_id = f"{prefix}{suffix}"

            try:
                # Try to fetch 1 note
                notes = client.get_notes(invitation=full_id, limit=1)
                
                if notes:
                    print(f"{year:<6} | v{api_version:<3} | \033[92mFOUND\033[0m      | {full_id}")
                    valid_invitations.append({
                        'year': year,
                        'api': api_version,
                        'invitation': full_id,
                        'count': len(notes) # Just confirms we got something
                    })
                else:
                    # No error, but no notes found (valid ID, empty content)
                    pass
            except openreview.OpenReviewException:
                # 404 Not Found or similar - invitation doesn't exist
                pass
            except Exception as e:
                print(f"{year:<6} | v{api_version:<3} | ERROR      | {full_id} ({str(e)})")

    return valid_invitations

if __name__ == "__main__":
    found = discover_iclr_invitations()
    
    print("\n\n--- Summary of Working Invitations ---")
    for item in found:
        print(f"Year {item['year']}: {item['invitation']}")